# Complete Squidpy Spatial Transcriptomics Workflow

Companion notebook for: [spatiabio.com — post 10](https://www.spatiabio.com/2026/07/complete-squidpy-spatial-transcriptomics.html)

Full pipeline combining all analyses from this series:
- Spatially variable genes (Moran's I)
- SVG visualization
- Neighborhood enrichment
- Co-occurrence
- Ligand-receptor interactions

In [ ]:
import squidpy as sq
import scanpy as sc
import warnings
warnings.filterwarnings('ignore')

## 1. Load data and preprocess

In [ ]:
adata = sq.datasets.visium_hne_adata()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='cell_ranger')
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, key_added='cluster')

sq.gr.spatial_neighbors(adata)
print(f'Spots: {adata.n_obs}, Clusters: {adata.obs["cluster"].nunique()}')

## 2. Spatially variable genes (Moran's I)

In [ ]:
sq.gr.spatial_autocorr(adata, mode='moran')
top_svgs = adata.uns['moranI'].head(5).index.tolist()
print('Top SVGs:', top_svgs)
sq.pl.spatial_scatter(adata, color=top_svgs[:2], ncols=2)

## 3. Neighborhood enrichment

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key='cluster')
sq.pl.nhood_enrichment(adata, cluster_key='cluster', title='Neighborhood enrichment')

## 4. Co-occurrence

In [ ]:
sq.gr.co_occurrence(adata, cluster_key='cluster')
top3 = adata.obs['cluster'].value_counts().head(3).index.tolist()
sq.pl.co_occurrence(adata, cluster_key='cluster', clusters=top3, figsize=(10, 4))

## 5. Ligand-receptor interactions

In [ ]:
sq.gr.ligrec(adata, cluster_key='cluster', n_perms=1000)
sq.pl.ligrec(
    adata,
    cluster_key='cluster',
    source_groups=top3,
    target_groups=top3,
    means_range=(0.3, float('inf')),
    alpha=0.001,
    swap_axes=True,
)